# Post-processed Yahoo CSVs

Loads each CSV via `yahoo_csv_postprocess` and shows shape, index, column names, and a preview.

**Kernel working directory:** project root (`finance_database/`) or this folder (`yahoo_data/`). The next cell fixes `import` paths and **`importlib.reload`s** `yahoo_csv_postprocess` so edits to that file show up without restarting the kernel.

**`yahoo_fundamentals_metrics.csv`:** output is long format (MultiIndex + `metric_source`, `metric_group`, `metric_name`, `metric_value`); the preview uses `reset_index()` so columns match `symbol`, `as_of_timestamp`, `metric_source`, `metric_group`, `metric_name`, `metric_value`, `source_url`, `status`.

**Reuse:** The load cell fills `df_dict` (e.g. `df_dict["df_yahoo_fundamentals_metrics"]`) and binds the same keys as variables (`df_yahoo_fundamentals_metrics`, `df_yahoo_timeseries`, ...).


In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython import get_ipython
from IPython.display import Markdown, display

_here = Path.cwd()
_candidates = [
    _here / "yahoo_data",
    _here,
    _here.parent / "yahoo_data",
]
for p in _candidates:
    if (p / "yahoo_csv_postprocess.py").is_file():
        sys.path.insert(0, str(p.resolve()))
        break

import importlib

import yahoo_csv_postprocess

importlib.reload(yahoo_csv_postprocess)
from yahoo_csv_postprocess import POSTPROCESS_BY_FILENAME

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

# How many rows to preview per file
HEAD_ROWS = 10

# Optional: restrict to these basenames, or None for all.
# You may omit the ``.csv`` suffix or pass a single string, e.g. ``"yahoo_key_statistics"``.
ONLY_FILES: list[str] | str | None = None

print("POSTPROCESS_BY_FILENAME:", sorted(POSTPROCESS_BY_FILENAME))

POSTPROCESS_BY_FILENAME: ['index_symbol_map.csv', 'yahoo_crawl_log.csv', 'yahoo_fundamentals_metrics.csv', 'yahoo_quote_metrics.csv', 'yahoo_timeseries.csv']


In [3]:
def show_df(name: str, df: pd.DataFrame, head_rows: int = HEAD_ROWS, *, var_name: str) -> None:
    display(Markdown(f"### {name} — variable `{var_name}`"))
    print(f"shape: {df.shape}")
    print(f"index: {df.index.names} ({type(df.index).__name__})")
    cols = list(df.columns)
    if len(cols) > 20:
        print(f"columns ({len(cols)}): {cols[:20]} ... (+{len(cols) - 20} more)")
    else:
        print(f"columns ({len(cols)}): {cols}")
    display(Markdown(f"**head({head_rows})**"))
    preview = df.sample(min(head_rows, len(df)))
    if name == "yahoo_fundamentals_metrics.csv":
        display(
            Markdown(
                "_Long format: index is `(symbol, as_of_timestamp, metric_source, metric_group, metric_name)`. "
                "Preview below uses `reset_index()` so each row shows all fields as columns._"
            )
        )
        flat = preview.reset_index()
        display(flat)
    elif name == "yahoo_key_statistics.csv":
        display(
            Markdown(
                "_Long format: index is `(symbol, timestamp, url, status)`. "
                "Preview below uses `reset_index()` so each row shows all fields as columns._"
            )
        )
        flat = preview.reset_index()
        display(flat)
    else:
        display(preview)
        

In [10]:
_registry = set(POSTPROCESS_BY_FILENAME)
if ONLY_FILES is None:
    names = sorted(_registry)
else:
    # Single string is a common mistake; treat as one filename
    _only = [ONLY_FILES] if isinstance(ONLY_FILES, str) else list(ONLY_FILES)
    wanted: set[str] = set()
    for f in _only:
        base = Path(f).name
        if not base.endswith(".csv"):
            base = f"{base}.csv"
        wanted.add(base)
    unknown = wanted - _registry
    if unknown:
        raise ValueError(
            f"Unknown or unregistered: {sorted(unknown)}. "
            f"Valid names: {sorted(_registry)}"
        )
    names = sorted(wanted)

df_dict: dict[str, pd.DataFrame] = {}
_ipy = get_ipython()
_user_ns = _ipy.user_ns if _ipy is not None else globals()

for csv_name in names:

    if csv_name != "yahoo_crawl_log.csv":

        var_name = f"df_{Path(csv_name).stem}"
        fn = POSTPROCESS_BY_FILENAME[csv_name]
        try:
            df = fn()
        except FileNotFoundError as e:
            display(Markdown(f"### {csv_name} (`{var_name}` not set)"))
            print(f"skip (file not found): {e}")
            continue
        except Exception as e:
            display(Markdown(f"### {csv_name} (`{var_name}` not set)"))
            print(f"ERROR: {e!r}")
            continue
        df_dict[var_name] = df
        _user_ns[var_name] = df
        show_df(csv_name, df, var_name=var_name)
        print()

print("df_dict keys:", sorted(df_dict))

### index_symbol_map.csv — variable `df_index_symbol_map`

shape: (111, 5)
index: ['symbol'] (Index)
columns (5): ['index_id', 'confidence', 'quoteType', 'matched_name', 'source_url']


**head(10)**

,index_id,confidence,quoteType,matched_name,source_url
symbol,,,,,
VGK,VGK,direct,etf,Vanguard FTSE Europe ETF,https://finance.yahoo.com/quote/VGK
ZC=F,ZC=F,direct,commodity,Corn Futures,https://finance.yahoo.com/quote/ZC=F
LLY,LLY,direct,equity,Eli Lilly and Company,https://finance.yahoo.com/quote/LLY
HD,HD,direct,equity,"Home Depot, Inc.",https://finance.yahoo.com/quote/HD
TCEHY,TCEHY,direct,equity,Tencent Holdings Limited,https://finance.yahoo.com/quote/TCEHY
^BSESN,^BSESN,direct,index,BSE Sensex,https://finance.yahoo.com/quote/^BSESN
MCD,MCD,direct,equity,McDonald's Corporation,https://finance.yahoo.com/quote/MCD
^HUI,^HUI,direct,index,NYSE Arca Gold BUGS Index,https://finance.yahoo.com/quote/^HUI
^FTSE,^FTSE,direct,index,FTSE 100 Index,https://finance.yahoo.com/quote/^FTSE


### yahoo_fundamentals_metrics.csv — variable `df_yahoo_fundamentals_metrics`

...

### yahoo_timeseries.csv — variable `df_yahoo_timeseries`

shape: (26593, 19)
index: ['symbol', 'date'] (MultiIndex)
columns (19): ['level_open', 'level_high', 'level_low', 'level_close', 'total_return_level', 'ma_50', 'ma_200', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_mid', 'bb_lower', 'stoch_k', 'stoch_d', 'source_url', 'source', 'technical_source_url']


**head(10)**

,,level_open,level_high,level_low,level_close,total_return_level,ma_50,...,bb_lower,stoch_k,stoch_d,source_url,source,technical_source_url
symbol,date,,,,,,,,,,,,,
SI=F,2009-06-01,15.85,15.94,13.44,13.57,NaN,11.65,...,9.045,38.65,40.68,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
ZC=F,2008-04-01,570.5,616,561.2,600.2,NaN,294.3,...,182,94.86,95.56,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
GILD,2019-07-01,68,69.35,64.16,65.52,NaN,78.99,...,59.12,26.96,24.83,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
^DJI,2012-12-01,1.303e+04,1.337e+04,1.288e+04,1.31e+04,NaN,1.105e+04,...,1.135e+04,77.05,80.05,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
HG=F,2024-04-01,4.074,4.691,4.044,4.564,NaN,3.668,...,3.311,89.01,68.22,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
AEDAUD=X,2022-09-30,0.4183,0.4409,0.4158,0.4252,NaN,0.3844,...,0.3402,81.29,82.96,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
IVV,2005-08-01,123.8,124.7,120.4,122.5,NaN,107.1,...,108.4,87.67,85.69,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
WMT,1992-11-01,5.073,5.49,4.875,5.427,NaN,1.449,...,0.07951,98.42,95.44,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
SI=F,2009-07-01,13.6,14.05,12.48,13.93,NaN,11.79,...,9.233,41.54,45.05,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...



df_dict keys: ['df_index_symbol_map', 'df_yahoo_fundamentals_metrics', 'df_yahoo_quote_metrics', 'df_yahoo_timeseries']


In [ ]:
# Export displayed DataFrames to individual CSV files.
# This uses the already-loaded df_dict from the previous cell.
from pathlib import Path

out_dir = Path.cwd() / "processed_csv"
out_dir.mkdir(parents=True, exist_ok=True)

if not df_dict:
    print("No DataFrames found in df_dict. Run the load/display cell first.")
else:
    written = 0
    for var_name, df in sorted(df_dict.items()):
        base_name = var_name[3:] if var_name.startswith("df_") else var_name
        out_path = out_dir / f"{base_name}.csv"
        df.reset_index().to_csv(out_path, index=False)
        written += 1
        print(f"wrote: {out_path}")

    print(f"Done. Exported {written} file(s) to {out_dir.resolve()}")